<a href="https://colab.research.google.com/github/tpiedrahita-byte/sys2025/blob/main/Pr%C3%A1ctica_2_Instrumentaci%C3%B3n_Electronica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica 2 - ESP32  
## Interrupciones, medición temporal, entradas analógicas y salidas PWM

### Integrantes
- Thomas Piedrahita Jaramillo
- Sebastian Montoya Bedoya

### Asignatura
Instrumentación Electrónica

### Plataforma de desarrollo
ESP32

### Entorno de programación
Arduino IDE

---

## Objetivo general

Comprender e implementar en el ESP32 el manejo de interrupciones, medición temporal entre eventos, entradas analógicas mediante conversión A/D y salidas PWM, visualizando los resultados mediante el Monitor Serial y el Serial Plotter del entorno Arduino IDE.

---

## Objetivos específicos

- Implementar el conteo de pulsos mediante interrupciones externas y filtrado por *debounce*.
- Medir el tiempo entre pulsaciones consecutivas y estimar la frecuencia asociada a dichos eventos.
- Adquirir señales analógicas provenientes de dos LDR, calcular su resistencia aproximada y determinar la dirección de una fuente de luz.
- Controlar la posición de un servomotor mediante PWM a partir de la diferencia de iluminación detectada por los sensores.
- Integrar sensado, procesamiento y actuación en una aplicación embebida coherente.

## Herramientas y materiales utilizados

Para el desarrollo de la práctica se utilizaron los siguientes elementos:

- Tarjeta de desarrollo **ESP32**
- **Arduino IDE**
- Pulsador
- Resistencias fijas de **10 kΩ**
- Dos fotorresistencias (**LDR**)
- Dos LEDs
- Un servomotor
- Protoboard y cables de conexión
- **Monitor Serial**
- **Serial Plotter**

También se consultó una guía de referencia del pinout del ESP32 para identificar correctamente las entradas y salidas del sistema.

## Estructura de la práctica

La práctica se organizó en cuatro actividades principales:

1. **Conteo de pulsos con interrupción y debounce**
2. **Medición del tiempo entre pulsos**
3. **Lectura de dos LDR mediante conversión A/D**
4. **Control de un servomotor mediante PWM**

En conjunto, la secuencia desarrollada puede resumirse así:

$$
\text{Detección} \rightarrow \text{Medición} \rightarrow \text{Sensado} \rightarrow \text{Actuación}
$$

Esto permitió pasar progresivamente desde la detección de eventos discretos hasta la implementación de un sistema embebido capaz de responder físicamente a una señal del entorno.

# Marco teórico general

## 1. Interrupciones

Una **interrupción** es un mecanismo mediante el cual el microcontrolador suspende momentáneamente la ejecución normal del programa para atender un evento. Este evento puede ser interno o externo, por ejemplo, un cambio de estado en un pin digital.

Su principal ventaja es que evita el uso constante de *polling*, permitiendo que el procesador solo atienda la señal cuando realmente ocurre el evento. Desde un punto de vista funcional, puede representarse como:

$$
\text{Evento en pin} \longrightarrow \text{Ejecución de ISR}
$$

donde ISR significa *Interrupt Service Routine* o rutina de servicio a la interrupción.

---

## 2. Rebote mecánico y debounce

Los pulsadores mecánicos no producen una transición perfecta entre estados lógicos. Durante una sola pulsación física pueden generarse varias transiciones rápidas, fenómeno conocido como **rebote**.

Si no se corrige, una única pulsación puede interpretarse como varios pulsos digitales. Para evitarlo se utiliza **debounce**, que puede ser por hardware o por software. En esta práctica se aplicó debounce por software, aceptando solo señales separadas por un tiempo mínimo.

Si $$t_n$$ es el tiempo actual y $$t_{n-1}$$ el tiempo de la última interrupción válida, entonces:

$$
\Delta t = t_n - t_{n-1}
$$

y el evento se acepta si:

$$
\Delta t > t_{\text{debounce}}
$$

## 3. Medición temporal entre eventos

Cuando se desea medir el tiempo entre dos eventos consecutivos, basta con registrar dos instantes y calcular su diferencia:

$$
T = t_2 - t_1
$$

Si el intervalo se expresa en microsegundos, puede estimarse una frecuencia equivalente mediante:

$$
f = \frac{10^6}{T_{\mu s}}
$$

Esta relación resulta útil para interpretar la rapidez con la que ocurren pulsos o pulsaciones.

---

## 4. Conversión analógico-digital

Una señal analógica puede tomar valores continuos dentro de un intervalo de voltaje. Para que el microcontrolador la procese, debe convertirse a un número digital por medio del ADC.

La relación general entre voltaje de entrada y lectura del ADC es:

$$
V_{in} = \frac{\text{ADC}}{N-1}V_{ref}
$$

En esta práctica se trabajó con el ESP32 en un rango de 0 a 4095 y una referencia aproximada de 3.3 V, por lo que:

$$
V_{in} = \frac{\text{ADC}}{4095}\cdot 3.3
$$

## 5. LDR y divisor de tensión

Una LDR es una resistencia dependiente de la luz. Su comportamiento general es:

- a mayor iluminación, menor resistencia
- a menor iluminación, mayor resistencia

Para medirla con el ESP32, cada LDR se conectó como parte de un divisor de tensión junto con una resistencia fija $$R_f$$ de 10 kΩ.

La ecuación del divisor es:

$$
V_{out} = V_{cc}\frac{R_f}{R_{LDR}+R_f}
$$

Despejando la resistencia de la LDR:

$$
R_{LDR} = R_f\left(\frac{V_{cc}}{V_{out}} - 1\right)
$$

Esta expresión permitió calcular una resistencia aproximada del sensor a partir del valor leído por el ADC.

---

## 6. Promedio móvil

Para suavizar las lecturas analógicas y reducir fluctuaciones, se empleó un promedio móvil de $$N$$ muestras:

$$
\overline{x}[n] = \frac{1}{N}\sum_{k=0}^{N-1}x[n-k]
$$

Este filtrado mejora la estabilidad de la señal y evita decisiones erráticas.

## 7. PWM y control de servomotores

La modulación por ancho de pulso, o **PWM**, consiste en generar una señal digital periódica cuyo tiempo en nivel alto cambia dentro de cada periodo.

Si $$T_{on}$$ es el tiempo en alto y $$T$$ el periodo total, entonces el ciclo útil es:

$$
D = \frac{T_{on}}{T}
$$

y el valor promedio ideal de la señal puede expresarse como:

$$
V_{prom} = D \cdot V_{cc}
$$

Aunque PWM no es una señal analógica real, sí permite controlar actuadores como LEDs, motores y servomotores.

En el caso del servomotor, la señal PWM se interpreta como una secuencia de pulsos cuyo ancho determina la posición angular del eje, por eso es adecuada para orientar un sistema según la información entregada por los sensores.

# Actividad 1  
## Conteo de pulsos con interrupción y debounce

### Objetivo específico

Implementar el conteo de pulsaciones válidas de un pulsador conectado al ESP32, usando una interrupción externa y debounce por software, mostrando el total acumulado en el Monitor Serial.

## Fundamento teórico de la Actividad 1

En esta actividad se trabajó con una interrupción externa activada por un flanco de bajada en el pin del pulsador. El botón se conectó usando la configuración `INPUT_PULLUP`, de manera que normalmente el pin se mantiene en nivel alto y, al presionar el pulsador, cambia a nivel bajo.

La interrupción se configuró con:

```cpp
attachInterrupt(digitalPinToInterrupt(pin), ISR, FALLING);

---



```markdown id="nb0010"
## Descripción del montaje y funcionamiento

En esta actividad se utilizó un pulsador conectado al **GPIO 27** del ESP32. El pin se configuró con resistencia pull-up interna, por lo que la pulsación se detectó mediante un flanco de bajada.

La lógica aplicada fue la siguiente:

1. ocurre una pulsación
2. se ejecuta la rutina de interrupción
3. se compara el tiempo actual con el tiempo de la última pulsación válida
4. si el tiempo supera el umbral de debounce, se incrementa el contador
5. el valor se muestra en el Monitor Serial

Esta actividad permitió estudiar el uso de interrupciones como mecanismo eficiente para detectar eventos discretos sin necesidad de consultar continuamente el estado del pin.

In [ ]:
const int buttonPin = 27;                 // GPIO del pulsador
volatile unsigned long pulseCount = 0;    // contador de pulsos válidos
volatile bool newPulse = false;           // bandera para avisar al loop
volatile unsigned long lastInterruptTime = 0;

const unsigned long debounceDelay = 150;  // ms

void IRAM_ATTR countPulse() {
  unsigned long currentTime = millis();

  if (currentTime - lastInterruptTime > debounceDelay) {
    pulseCount++;
    newPulse = true;
    lastInterruptTime = currentTime;
  }
}

void setup() {
  Serial.begin(115200);

  pinMode(buttonPin, INPUT_PULLUP);

  attachInterrupt(digitalPinToInterrupt(buttonPin), countPulse, FALLING);

  Serial.println("Conteo de pulsos con interrupcion y debounce");
  Serial.println("Presiona el pulsador...");
}

void loop() {
  if (newPulse) {
    noInterrupts();
    unsigned long currentCount = pulseCount;
    newPulse = false;
    interrupts();

    Serial.print("Pulsos contados: ");
    Serial.println(currentCount);
  }
}

## Análisis de la Actividad 1

Esta actividad permitió comprobar que las interrupciones son una herramienta muy útil para detectar eventos discretos de manera eficiente. El ESP32 no necesita estar leyendo constantemente el estado del botón, sino que responde únicamente cuando se produce la transición configurada.

Además, se observó que el *debounce* por software es indispensable cuando se trabaja con pulsadores mecánicos. Sin esta validación temporal, una sola pulsación podría generar múltiples conteos incorrectos.

También se aplicó una buena práctica de programación embebida: la ISR solo actualiza variables esenciales, mientras que la impresión serial se realiza en el `loop()`.

# Actividad 2  
## Medición del tiempo entre pulsos

### Objetivo específico

Medir el tiempo transcurrido entre pulsaciones consecutivas de un botón usando interrupciones externas y la función `micros()`, mostrando el intervalo en microsegundos y una frecuencia estimada en el Monitor Serial.

## Fundamento teórico de la Actividad 2

En esta actividad no solo se detectó la pulsación, sino que además se registró el instante exacto en que ocurrió cada evento para calcular el tiempo entre pulsos consecutivos.

Si $$t_k$$ y $$t_{k-1}$$ son dos instantes consecutivos de pulsación, entonces el intervalo entre ellos es:

$$
T_k = t_k - t_{k-1}
$$

Como la función `micros()` entrega el tiempo en microsegundos, la frecuencia estimada puede calcularse mediante:

$$
f_k = \frac{10^6}{T_k}
$$

La primera pulsación no permite calcular un intervalo, por lo que solo se utiliza como referencia inicial. A partir de la segunda, sí se obtiene el tiempo entre eventos.

También se aplicó debounce por software, esta vez usando una ventana de:

$$
t_{\text{debounce}} = 150000 \ \mu s
$$

## Descripción del montaje y funcionamiento

Se reutilizó el mismo pulsador conectado al **GPIO 27**. La interrupción se disparó igualmente con flanco de bajada.

La lógica aplicada fue:

1. la primera pulsación guarda una referencia temporal
2. la segunda y las siguientes calculan el tiempo transcurrido desde la pulsación anterior
3. el valor del intervalo se imprime en microsegundos
4. además se estima la frecuencia equivalente en hertz

Esta actividad permitió pasar del simple conteo de eventos a la medición temporal precisa entre ellos.

In [ ]:
const int buttonPin = 27;   // mismo pin del punto 1

volatile unsigned long lastPulseTime = 0;     // tiempo del pulso anterior
volatile unsigned long interval = 0;          // tiempo entre pulsos
volatile bool newData = false;                // avisa al loop que hay dato nuevo
volatile bool firstPulse = true;              // para ignorar la primera medición

volatile unsigned long lastDebounceTime = 0;  // para debounce por software
const unsigned long debounceDelay = 150000;   // 150 ms en microsegundos

void IRAM_ATTR handleInterrupt() {
  unsigned long currentTime = micros();

  // Debounce por software
  if (currentTime - lastDebounceTime > debounceDelay) {

    if (firstPulse) {
      // Primera pulsación: solo guarda referencia
      lastPulseTime = currentTime;
      firstPulse = false;
    } else {
      // A partir de la segunda: calcular intervalo
      interval = currentTime - lastPulseTime;
      lastPulseTime = currentTime;
      newData = true;
    }

    lastDebounceTime = currentTime;
  }
}

void setup() {
  Serial.begin(115200);

  pinMode(buttonPin, INPUT_PULLUP);

  attachInterrupt(digitalPinToInterrupt(buttonPin), handleInterrupt, FALLING);

  Serial.println("PUNTO 2 - Medidor de tiempo entre pulsos con interrupciones en ESP32");
  Serial.println("Montaje: boton entre GPIO27 y GND");
  Serial.println("La primera pulsacion solo inicializa la referencia.");
  Serial.println("Desde la segunda pulsacion se mostrara el tiempo entre pulsos.");
}

void loop() {
  if (newData) {
    noInterrupts();
    unsigned long localInterval = interval;
    newData = false;
    interrupts();

    Serial.print("Tiempo entre pulsos: ");
    Serial.print(localInterval);
    Serial.println(" us");

    if (localInterval > 0) {
      float freq = 1000000.0 / localInterval;
      Serial.print("Frecuencia estimada: ");
      Serial.print(freq, 2);
      Serial.println(" Hz");
    }

    Serial.println("---------------------------");
  }
}

## Análisis de la Actividad 2

En esta actividad se comprobó que el ESP32 puede medir el tiempo entre eventos de manera precisa usando interrupciones y la función `micros()`.

La resolución temporal en microsegundos permitió relacionar directamente el intervalo entre pulsaciones con una frecuencia estimada. Esto resulta útil cuando se desea caracterizar la rapidez de ocurrencia de eventos discretos.

También fue importante utilizar la primera pulsación solo como referencia inicial, ya que no es posible medir un intervalo sin un evento previo.

# Actividad 3  
## Lectura de dos LDR mediante conversión A/D

### Objetivo específico

Leer dos LDR mediante las entradas analógicas del ESP32, calcular la resistencia aproximada de cada sensor, determinar la dirección del foco de luz y activar una alarma visual con dos LEDs.

## Fundamento teórico de la Actividad 3

En esta actividad se trabajó con dos LDR conectadas como divisores de tensión. Cada sensor se conectó de la siguiente forma:

$$
3.3V \rightarrow LDR \rightarrow V_{out} \rightarrow R_f \rightarrow GND
$$

donde:

$$
R_f = 10\,k\Omega
$$

A partir de la lectura del ADC se calculó el voltaje en el nodo intermedio mediante:

$$
V = \frac{\text{ADC}}{4095}\cdot 3.3
$$

Luego, usando la ecuación del divisor de tensión, se estimó la resistencia de cada LDR:

$$
R_{LDR} = R_f\left(\frac{V_{cc}}{V} - 1\right)
$$

La dirección del foco de luz se determinó comparando las dos lecturas. Para ello se calcularon proporciones relativas:

$$
r_L = \frac{ADC_L}{ADC_L + ADC_R}
$$

$$
r_R = \frac{ADC_R}{ADC_L + ADC_R}
$$

y se definió un umbral de decisión:

$$
\delta = 0.08
$$

De este modo:

$$
r_L - r_R > \delta \Rightarrow \text{IZQUIERDA}
$$

$$
r_R - r_L > \delta \Rightarrow \text{DERECHA}
$$

$$
|r_L - r_R| \leq \delta \Rightarrow \text{CENTRO}
$$

## Suavizado y validación de luz mínima

Para mejorar la estabilidad de la lectura se utilizó un promedio móvil de 8 muestras:

$$
\overline{x}[n] = \frac{1}{8}\sum_{k=0}^{7}x[n-k]
$$

Este promedio reduce el efecto del ruido y hace más robusta la decisión sobre la posición del foco.

Además, para evitar decisiones erróneas cuando la iluminación era demasiado baja, se impuso la condición:

$$
ADC_L + ADC_R < 80 \Rightarrow \text{sin dirección válida}
$$

En ese caso, ambos LEDs permanecen apagados y el sistema informa que no se puede determinar la dirección de la luz.

## Descripción del montaje y funcionamiento

En esta actividad se usaron los siguientes pines:

- **GPIO35**: LDR izquierda
- **GPIO34**: LDR derecha
- **GPIO25**: LED izquierdo
- **GPIO26**: LED derecho

El sistema realiza estos pasos:

1. adquiere las lecturas analógicas de ambas LDR
2. suaviza las señales mediante promedio móvil
3. convierte el valor ADC a voltaje
4. calcula la resistencia aproximada de cada sensor
5. compara ambos sensores para determinar la posición del foco
6. enciende un LED según la dirección detectada
7. muestra toda la información en el Monitor Serial o el Serial Plotter

Esta actividad fue la primera en la que el sistema comenzó a sensar una variable física del entorno: la intensidad luminosa.

In [ ]:
/*
  PUNTO 3 - Entradas con conversion A/D en ESP32
  Lectura de 2 LDR, calculo de resistencia, deteccion de direccion de luz
  y alarma visual con 2 LEDs.

  Conexion de cada LDR:
  3.3V ---- LDR ---- nodo_ADC ---- R_fija(10k) ---- GND

  Con esta conexion:
  - mas luz  -> menor R_LDR
  - mas luz  -> mayor Vout
  - mas luz  -> mayor ADC
*/

#define LDR_LEFT_PIN    35   // ADC1
#define LDR_RIGHT_PIN   34   // ADC1

#define LED_LEFT_PIN    25
#define LED_RIGHT_PIN   26

#define R_FIXED         10000.0   // resistencia fija del divisor (ohm)
#define VCC             3.3
#define ADC_MAX         4095.0

#define N_SMOOTH        8         // tamaño del promedio movil
#define MIN_ADC_SUM     80.0      // umbral de luz minima para considerar señal valida
#define DIFF_THRESHOLD  0.08      // banda muerta relativa para decidir izquierda/centro/derecha

#define USE_PLOTTER     0         // 0 = texto para monitor serial, 1 = salida para serial plotter

float bufferLeft[N_SMOOTH];
float bufferRight[N_SMOOTH];
int idxSmooth = 0;
bool filled = false;

void setup() {
  Serial.begin(115200);

  pinMode(LDR_LEFT_PIN, INPUT);
  pinMode(LDR_RIGHT_PIN, INPUT);

  pinMode(LED_LEFT_PIN, OUTPUT);
  pinMode(LED_RIGHT_PIN, OUTPUT);

  digitalWrite(LED_LEFT_PIN, LOW);
  digitalWrite(LED_RIGHT_PIN, LOW);

  for (int i = 0; i < N_SMOOTH; i++) {
    bufferLeft[i] = 0.0;
    bufferRight[i] = 0.0;
  }

  Serial.println("=== PUNTO 3: Direccion de luz con 2 LDRs en ESP32 ===");
  Serial.println("Monitor serial a 115200 baudios");
  Serial.println("LDR izquierda en GPIO35, LDR derecha en GPIO34");
  Serial.println("LED izquierda en GPIO25, LED derecha en GPIO26");
  delay(500);
}

float readSmooth(int pin, float *buffer) {
  int adc = analogRead(pin);
  buffer[idxSmooth] = (float)adc;

  float sum = 0.0;
  int count = filled ? N_SMOOTH : (idxSmooth + 1);

  for (int i = 0; i < count; i++) {
    sum += buffer[i];
  }

  return sum / count;
}

float adcToVoltage(float adcValue) {
  return (adcValue / ADC_MAX) * VCC;
}

float voltageToLdrResistance(float vout) {
  // Evitar division por cero o valores extremos
  if (vout <= 0.001) return 1000000.0;   // 1 megaohm como valor maximo de referencia
  if (vout >= (VCC - 0.001)) return 1.0; // muy iluminado, resistencia muy baja aprox.

  return R_FIXED * ((VCC / vout) - 1.0);
}

void loop() {
  float adcLeft = readSmooth(LDR_LEFT_PIN, bufferLeft);
  float adcRight = readSmooth(LDR_RIGHT_PIN, bufferRight);

  idxSmooth++;
  if (idxSmooth >= N_SMOOTH) {
    idxSmooth = 0;
    filled = true;
  }

  float sumAdc = adcLeft + adcRight;

  // muy poca luz
  if (sumAdc < MIN_ADC_SUM) {
    digitalWrite(LED_LEFT_PIN, LOW);
    digitalWrite(LED_RIGHT_PIN, LOW);

    #if USE_PLOTTER
      Serial.println("0\t0\t0\t0");
    #else
      Serial.println("Muy poca luz -> no se determina direccion.");
    #endif

    delay(150);
    return;
  }

  float vLeft = adcToVoltage(adcLeft);
  float vRight = adcToVoltage(adcRight);

  float rLeft = voltageToLdrResistance(vLeft);
  float rRight = voltageToLdrResistance(vRight);

  // Como mayor ADC significa mas luz, usamos una proporcion de comparacion
  float ratioRight = adcRight / sumAdc;   // 0 a 1
  float ratioLeft  = adcLeft  / sumAdc;   // 0 a 1

  String position;

  if ((ratioLeft - ratioRight) > DIFF_THRESHOLD) {
    position = "IZQUIERDA";
    digitalWrite(LED_LEFT_PIN, HIGH);
    digitalWrite(LED_RIGHT_PIN, LOW);
  }
  else if ((ratioRight - ratioLeft) > DIFF_THRESHOLD) {
    position = "DERECHA";
    digitalWrite(LED_LEFT_PIN, LOW);
    digitalWrite(LED_RIGHT_PIN, HIGH);
  }
  else {
    position = "CENTRO";
    digitalWrite(LED_LEFT_PIN, LOW);
    digitalWrite(LED_RIGHT_PIN, LOW);
  }

  #if USE_PLOTTER
    // Formato amigable para serial plotter
    // adcLeft    adcRight   ledLeft ledRight
    Serial.print(adcLeft); Serial.print("\t");
    Serial.print(adcRight); Serial.print("\t");
    Serial.print(digitalRead(LED_LEFT_PIN) ? 4095 : 0); Serial.print("\t");
    Serial.println(digitalRead(LED_RIGHT_PIN) ? 4095 : 0);
  #else
    Serial.print("ADC_L: ");
    Serial.print(adcLeft, 1);
    Serial.print(" | V_L: ");
    Serial.print(vLeft, 3);
    Serial.print(" V | R_LDR_L: ");
    Serial.print(rLeft, 1);
    Serial.print(" ohm");

    Serial.print(" || ADC_R: ");
    Serial.print(adcRight, 1);
    Serial.print(" | V_R: ");
    Serial.print(vRight, 3);
    Serial.print(" V | R_LDR_R: ");
    Serial.print(rRight, 1);
    Serial.print(" ohm");

    Serial.print(" || Posicion del foco: ");
    Serial.println(position);
  #endif

  delay(150);
}

## Análisis de la Actividad 3

Esta actividad permitió verificar que el ESP32 puede adquirir señales analógicas del entorno y convertirlas en información útil para la toma de decisiones.

Las lecturas del ADC se tradujeron en voltaje y posteriormente en una estimación de resistencia para cada LDR, lo cual dio una interpretación física más clara del comportamiento del sistema.

La comparación entre ambos sensores permitió determinar la dirección de una fuente de luz, mientras que los LEDs ofrecieron una respuesta visual inmediata y fácil de validar experimentalmente.

# Actividad 4  
## Control de un servomotor mediante PWM

### Objetivo específico

Implementar una salida PWM en el ESP32 para controlar la posición de un servomotor en función de la dirección del foco de luz detectado por dos LDR, mostrando además la resistencia estimada de cada sensor y el ángulo asignado al servo.

## Fundamento teórico de la Actividad 4

En esta actividad se integró la información de las dos LDR para controlar un servomotor mediante PWM.

Primero se definió un **contraste normalizado** entre ambas lecturas:

$$
C = \frac{ADC_L - ADC_R}{ADC_L + ADC_R}
$$

Su interpretación es:

- $$C > 0$$: más luz a la izquierda
- $$C < 0$$: más luz a la derecha
- $$C \approx 0$$: luz centrada

Para evitar vibraciones del servo debidas a pequeñas diferencias entre sensores, se implementó una zona muerta:

$$
|C| < 0.06 \Rightarrow C = 0
$$

Luego, si $$\theta_{min}$$ y $$\theta_{max}$$ representan los límites del servo, y $$\theta_c$$ su posición central, el ángulo aplicado se calculó como:

$$
\theta_c = \frac{\theta_{min} + \theta_{max}}{2}
$$

$$
\theta = \theta_c + C\cdot \frac{\theta_{max}-\theta_{min}}{2}\cdot K
$$

donde $$K$$ es una ganancia de sensibilidad. En esta práctica se usó:

$$
\theta_{min}=0^\circ,\qquad \theta_{max}=180^\circ,\qquad K=1
$$

## Condición de baja iluminación y lógica general

Para evitar movimientos erráticos cuando la iluminación era demasiado baja, se definió una condición de seguridad:

$$
ADC_L + ADC_R < 80 \Rightarrow \theta = \theta_c
$$

Es decir, si la suma de lecturas era muy pequeña, el sistema recentraba el servo y apagaba los LEDs, indicando que no existía información suficiente para estimar una dirección confiable.

La lógica general de esta actividad fue:

1. leer las dos LDR
2. suavizar las lecturas
3. calcular voltajes y resistencias
4. obtener el contraste entre sensores
5. aplicar zona muerta
6. calcular el ángulo del servo
7. mover el actuador y encender el LED correspondiente

Con esta etapa se completó un sistema embebido capaz de sensar, procesar y actuar.

## Descripción del montaje y funcionamiento

En esta actividad se utilizaron los siguientes pines:

- **GPIO35**: LDR izquierda
- **GPIO34**: LDR derecha
- **GPIO25**: LED izquierdo
- **GPIO26**: LED derecho
- **GPIO32**: señal del servomotor

El servomotor se alimentó externamente y compartió tierra con el ESP32.

El sistema interpreta hacia qué lado hay mayor concentración de luz y orienta el servo en consecuencia. Si la iluminación se concentra en el lado izquierdo, el ángulo aumenta hacia esa dirección; si se concentra en el lado derecho, el ángulo se corrige en sentido contrario.

In [ ]:
#include <ESP32Servo.h>
#include <math.h>

/*
  PUNTO 4 - Rastreador de luz con 2 LDR y servomotor en ESP32

  LDR izquierda:
  3.3V ---- LDR ---- nodo ---- 10k ---- GND
                     |
                   GPIO35

  LDR derecha:
  3.3V ---- LDR ---- nodo ---- 10k ---- GND
                     |
                   GPIO34

  LEDs:
  GPIO25 -> LED izquierdo
  GPIO26 -> LED derecho

  Servo:
  Señal -> GPIO32
  Vcc   -> 5V
  GND   -> GND comun con ESP32
*/

// ---------------- PINES ----------------
#define LDR_LEFT_PIN      35
#define LDR_RIGHT_PIN     34
#define LED_LEFT_PIN      25
#define LED_RIGHT_PIN     26
#define SERVO_PIN         32

// ---------------- CONSTANTES ----------------
#define R_FIXED           10000.0     // 10k ohm
#define VCC_ADC           3.3
#define ADC_MAX           4095.0

#define N_SMOOTH          8           // promedio movil
#define MIN_ADC_SUM       80.0        // umbral minimo de luz util
#define DEAD_ZONE         0.06        // zona muerta para evitar vibracion
#define SERVO_GAIN        1.0         // sensibilidad del seguimiento

// Cambia a 90 si tu servo es de 90 grados
#define SERVO_MIN_ANGLE   0
#define SERVO_MAX_ANGLE   180

#define USE_PLOTTER       0           // 0 = monitor serial, 1 = serial plotter

// ---------------- VARIABLES ----------------
Servo servo;

float bufferLeft[N_SMOOTH];
float bufferRight[N_SMOOTH];
int idxSmooth = 0;
bool filled = false;

const int servoCenter = (SERVO_MIN_ANGLE + SERVO_MAX_ANGLE) / 2;
const int servoHalfRange = (SERVO_MAX_ANGLE - SERVO_MIN_ANGLE) / 2;

// ---------------- FUNCIONES ----------------
float readSmooth(int pin, float *buffer) {
  int adc = analogRead(pin);
  buffer[idxSmooth] = (float)adc;

  float sum = 0.0;
  int count = filled ? N_SMOOTH : (idxSmooth + 1);

  for (int i = 0; i < count; i++) {
    sum += buffer[i];
  }

  return sum / count;
}

float adcToVoltage(float adcValue) {
  return (adcValue / ADC_MAX) * VCC_ADC;
}

float voltageToLdrResistance(float vout) {
  // Protecciones para evitar division por cero
  if (vout <= 0.001) return 1000000.0;   // ~1 megaohm
  if (vout >= (VCC_ADC - 0.001)) return 1.0;

  return R_FIXED * ((VCC_ADC / vout) - 1.0);
}

void setup() {
  Serial.begin(115200);

  pinMode(LDR_LEFT_PIN, INPUT);
  pinMode(LDR_RIGHT_PIN, INPUT);

  pinMode(LED_LEFT_PIN, OUTPUT);
  pinMode(LED_RIGHT_PIN, OUTPUT);

  digitalWrite(LED_LEFT_PIN, LOW);
  digitalWrite(LED_RIGHT_PIN, LOW);

  for (int i = 0; i < N_SMOOTH; i++) {
    bufferLeft[i] = 0.0;
    bufferRight[i] = 0.0;
  }

  servo.setPeriodHertz(50);                  // frecuencia tipica de servo
  servo.attach(SERVO_PIN, 500, 2400);        // min/max pulse width
  servo.write(servoCenter);

  Serial.println("=== PUNTO 4: Rastreador de luz con servo en ESP32 ===");
  Serial.println("LDR izquierda -> GPIO35");
  Serial.println("LDR derecha   -> GPIO34");
  Serial.println("Servo         -> GPIO32");
  Serial.println("LED izq       -> GPIO25");
  Serial.println("LED der       -> GPIO26");
  Serial.println("Servo centrado al iniciar");
  delay(1000);
}

void loop() {
  float adcLeft = readSmooth(LDR_LEFT_PIN, bufferLeft);
  float adcRight = readSmooth(LDR_RIGHT_PIN, bufferRight);

  idxSmooth++;
  if (idxSmooth >= N_SMOOTH) {
    idxSmooth = 0;
    filled = true;
  }

  float sumAdc = adcLeft + adcRight;

  // Si hay muy poca luz, mantener centrado
  if (sumAdc < MIN_ADC_SUM) {
    servo.write(servoCenter);
    digitalWrite(LED_LEFT_PIN, LOW);
    digitalWrite(LED_RIGHT_PIN, LOW);

    #if USE_PLOTTER
      Serial.println("0\t0\t0");
    #else
      Serial.println("Muy poca luz -> servo centrado, no se estima direccion.");
    #endif

    delay(150);
    return;
  }

  // Conversion a voltaje
  float vLeft = adcToVoltage(adcLeft);
  float vRight = adcToVoltage(adcRight);

  // Resistencia estimada de cada LDR
  float rLeft = voltageToLdrResistance(vLeft);
  float rRight = voltageToLdrResistance(vRight);

  // Contraste normalizado con ADC:
  // positivo -> mas luz a la izquierda
  // negativo -> mas luz a la derecha
  float contrast = (adcLeft - adcRight) / sumAdc;

  // Banda muerta
  if (fabs(contrast) < DEAD_ZONE) {
    contrast = 0.0;
  }

  // Estimacion del angulo del servo
  int angle = servoCenter + (int)(contrast * servoHalfRange * SERVO_GAIN);

  angle = constrain(angle, SERVO_MIN_ANGLE, SERVO_MAX_ANGLE);
  servo.write(angle);

  // LEDs de direccion
  if (contrast > 0.0) {
    digitalWrite(LED_LEFT_PIN, HIGH);
    digitalWrite(LED_RIGHT_PIN, LOW);
  } else if (contrast < 0.0) {
    digitalWrite(LED_LEFT_PIN, LOW);
    digitalWrite(LED_RIGHT_PIN, HIGH);
  } else {
    digitalWrite(LED_LEFT_PIN, LOW);
    digitalWrite(LED_RIGHT_PIN, LOW);
  }

  #if USE_PLOTTER
    // Resistencia izquierda, resistencia derecha, angulo
    Serial.print(rLeft);
    Serial.print("\t");
    Serial.print(rRight);
    Serial.print("\t");
    Serial.println(angle);
  #else
    Serial.print("R_LDR_IZQ: ");
    Serial.print(rLeft, 1);
    Serial.print(" ohm | R_LDR_DER: ");
    Serial.print(rRight, 1);
    Serial.print(" ohm | Contraste: ");
    Serial.print(contrast, 3);
    Serial.print(" | Angulo servo: ");
    Serial.println(angle);
  #endif

  delay(150);
}

## Análisis de la Actividad 4

Esta fue la actividad más completa de la práctica, ya que integró sensado, procesamiento y actuación en un mismo sistema.

El ESP32 dejó de ser únicamente un observador de señales para convertirse en un sistema capaz de responder físicamente al entorno. Las dos LDR entregan información sobre la distribución espacial de la luz, el microcontrolador procesa esa información y el servomotor reacciona orientándose según el lado de mayor iluminación.

La zona muerta evitó vibraciones innecesarias y el promedio móvil hizo más estable el seguimiento, especialmente ante pequeñas fluctuaciones del ADC.

# Discusión general

La práctica estuvo estructurada de manera progresiva y coherente. En la primera actividad se estudió la detección de eventos discretos mediante interrupciones. En la segunda, esos eventos se analizaron desde el punto de vista temporal. En la tercera, se incorporó el tratamiento de señales analógicas provenientes del entorno. Finalmente, en la cuarta actividad, esa información se utilizó para controlar un actuador real.

En términos funcionales, la secuencia completa puede resumirse como:

$$
\text{Detección} \rightarrow \text{Medición} \rightarrow \text{Sensado} \rightarrow \text{Actuación}
$$

Cada actividad aportó una competencia distinta:

- manejo de ISR y variables compartidas
- medición temporal de eventos
- interpretación de señales analógicas
- toma de decisiones a partir de sensores
- control de actuadores mediante PWM

Esto permitió construir una visión integral del comportamiento de un sistema embebido aplicado a instrumentación electrónica.

# Conclusiones generales

A partir del desarrollo de esta práctica se concluye que el ESP32 es una plataforma adecuada para integrar distintas funciones fundamentales de los sistemas embebidos.

En la **Actividad 1** se comprobó que las interrupciones permiten detectar pulsos de forma eficiente y que el debounce por software es indispensable al trabajar con pulsadores mecánicos.

En la **Actividad 2** se evidenció que la función `micros()` permite medir con precisión el tiempo entre eventos y relacionarlo con una frecuencia estimada.

En la **Actividad 3** se demostró que las entradas analógicas del ESP32 pueden utilizarse para adquirir información física del entorno, calcular magnitudes eléctricas asociadas y tomar decisiones sobre la dirección de una fuente de luz.

En la **Actividad 4** se logró cerrar el ciclo completo de un sistema embebido, usando la información sensada para controlar un servomotor mediante PWM y producir una respuesta física coherente.

En conjunto, la práctica integró teoría y aplicación, mostrando que un sistema embebido puede detectar, medir, procesar y actuar de manera automática frente a su entorno.